# Titanic Survival Prediction
Who survived the sinking of the Titanic? Using the passenger list (class, sex, age, fare, family aboard…), this notebook engineers features, compares models, and builds a pipeline that can score *any* passenger profile a person types into a form.

**Data:** 891 passengers (Kaggle training set), 38% survived.  
**Metric:** ROC-AUC, plus accuracy / F1.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay
from train import DATA_URL, fetch
from features import FamilyFeatures   # pipeline step: family size, alone flag, log fare per person
sns.set_theme(style='whitegrid'); SEED = 42

## 1. Load

In [2]:
df = pd.read_csv(fetch(DATA_URL, Path('data') / 'titanic.csv'))
print(df.shape)
print(df.isna().sum()[lambda s: s > 0])
df.head()

(891, 12)
Age         177
Cabin       687
Embarked      2
dtype: int64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


`Cabin` is 77% missing and `Ticket` is a free-text code, so both are dropped. `Age` is missing for 177 passengers (handled below), `Embarked` for 2 (filled with the most common port, Southampton).

## 2. Feature engineering
**Title from the name.** `"Braund, Mr. Owen Harris"` → `Mr`. The title encodes sex, age group (Master = boy) and social status in one field. French forms are mapped to English ones and rare titles grouped.

In [3]:
TITLES = {'Mr': 'Mr', 'Mrs': 'Mrs', 'Miss': 'Miss', 'Master': 'Master', 'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'}
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')[0].map(TITLES).fillna('Rare')
df['Pclass'] = df['Pclass'].map({1: '1st', 2: '2nd', 3: '3rd'})
df['Embarked'] = df['Embarked'].fillna('S').map({'S': 'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'})
df.groupby('Title')['Survived'].agg(['mean', 'size']).round(2)

,mean,size
Title,,
Master,0.57,40
Miss,0.70,185
Mr,0.16,517
Mrs,0.79,126
Rare,0.35,23


**Family features.** `SibSp` (siblings/spouses) + `Parch` (parents/children) + 1 = family size. The fare is paid per ticket, which is often shared by a family, so fare per person is a cleaner wealth signal.

In [4]:
eda = df.assign(family_size=df.SibSp + df.Parch + 1)
eda['is_alone'] = eda.family_size == 1
eda['fare_per_person'] = eda.Fare / eda.family_size
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.barplot(data=eda, x='Pclass', y='Survived', hue='Sex', order=['1st', '2nd', '3rd'], errorbar=None, ax=axes[0])
eda.groupby(eda.family_size.clip(upper=8))['Survived'].mean().plot.bar(ax=axes[1], color='#4f46e5', rot=0)
axes[1].set_title('Survival by family size (8 = 8+)')
sns.kdeplot(data=eda[eda.fare_per_person > 0], x='fare_per_person', hue='Survived', common_norm=False, fill=True, log_scale=True, ax=axes[2])
axes[0].set_title('Survival by class and sex'); plt.tight_layout()

**Findings**
- Women survived far more than men in every class; 1st-class women almost all survived, 3rd-class men rarely did.
- Small families (2–4) did best; people travelling alone and very large families did worst.
- Higher fare per person → higher survival (cabins closer to the boat deck, priority).

**Missing ages.** Instead of one global median, each missing age gets the median age of passengers with the same title (a missing-age 'Master' is a child, not 28). The medians are learned **on the training split only**.

In [5]:
FEATURES = ['Pclass', 'Sex', 'Age', 'Fare', 'SibSp', 'Parch', 'Embarked', 'Title']
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['Survived'], random_state=SEED)
age_by_title = train_df.groupby('Title')['Age'].median()
print(age_by_title)
for part in (train_df, test_df):
    part['Age'] = part['Age'].fillna(part['Title'].map(age_by_title))
X_train, y_train = train_df[FEATURES], train_df['Survived']
X_test, y_test = test_df[FEATURES], test_df['Survived']

Title
Master     3.0
Miss      22.0
Mr        30.0
Mrs       35.0
Rare      49.0
Name: Age, dtype: float64


## 3. Modelling
The model only takes inputs a person can type into a form (class, sex, age, fare, siblings/spouses, parents/children, port, title). The engineered family features are computed **inside the pipeline** by a small custom transformer (`features.py`), so the browser demo computes exactly the same features.

First: do the engineered features help? A Logistic Regression with and without them:

In [6]:
def make_prep(scale, engineered=True):
    num = StandardScaler() if scale else 'passthrough'
    if engineered:
        fam = Pipeline([('fe', FamilyFeatures()), ('sc', StandardScaler())]) if scale else FamilyFeatures()
    else:
        fam = num
    return ColumnTransformer([('age', num, ['Age']), ('family', fam, ['SibSp', 'Parch', 'Fare']),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), ['Pclass', 'Sex', 'Embarked', 'Title'])])

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
for eng in (False, True):
    lr = Pipeline([('prep', make_prep(True, eng)), ('model', LogisticRegression(max_iter=2000, C=0.5))])
    r = cross_validate(lr, X_train, y_train, cv=cv, scoring=['roc_auc', 'accuracy'])
    print(f"engineered={eng!s:5s}  ROC-AUC={r['test_roc_auc'].mean():.3f}  accuracy={r['test_accuracy'].mean():.3f}")

engineered=False  ROC-AUC=0.870  accuracy=0.823


engineered=True   ROC-AUC=0.873  accuracy=0.823


The engineered family features give only a small gain for the linear model (title and sex already carry most of what family size says). They are kept because they cost nothing at prediction time and let the tree models use family size directly.

Now the model comparison (5×3 repeated stratified CV on the training set):

In [7]:
models = {
    'Logistic Regression': Pipeline([('prep', make_prep(True)), ('model', LogisticRegression(max_iter=2000, C=0.5))]),
    'Random Forest': Pipeline([('prep', make_prep(False)), ('model', RandomForestClassifier(
        n_estimators=400, max_depth=7, min_samples_leaf=3, random_state=SEED, n_jobs=-1))]),
    'Gradient Boosting': Pipeline([('prep', make_prep(False)), ('model', GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=SEED))]),
}
rows = []
for name, m in models.items():
    r = cross_validate(m, X_train, y_train, cv=cv, scoring=['roc_auc', 'f1', 'accuracy'])
    rows.append({'model': name, **{k: r[f'test_{k}'].mean() for k in ['roc_auc', 'f1', 'accuracy']},
                 'roc_auc_std': r['test_roc_auc'].std()})
pd.DataFrame(rows).sort_values('roc_auc', ascending=False).round(3)

,model,roc_auc,f1,accuracy,roc_auc_std
2,Gradient Boosting,0.888,0.768,0.832,0.026
1,Random Forest,0.881,0.766,0.831,0.024
0,Logistic Regression,0.873,0.762,0.823,0.023


Gradient Boosting is slightly ahead (it can capture interactions such as *sex × class* without help); the gap to Logistic Regression is small compared with the fold-to-fold spread.

## 4. Final evaluation on the test set

In [8]:
best = models['Gradient Boosting'].fit(X_train, y_train)
proba = best.predict_proba(X_test)[:, 1]
print('Test ROC-AUC:', round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, proba >= 0.5, target_names=['Died', 'Survived']))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, proba >= 0.5, display_labels=['Died', 'Survived'],
                                        cmap='Blues', colorbar=False, ax=axes[0])
RocCurveDisplay.from_predictions(y_test, proba, name='Gradient Boosting', ax=axes[1])
axes[1].plot([0, 1], [0, 1], '--', color='grey'); plt.tight_layout()

Test ROC-AUC: 0.847
              precision    recall  f1-score   support

        Died       0.82      0.89      0.85       110
    Survived       0.80      0.68      0.73        69

    accuracy                           0.81       179
   macro avg       0.81      0.79      0.79       179
weighted avg       0.81      0.81      0.81       179



The test score is a bit below the CV average — with 179 test passengers, a few hard cases move it by several points. CV is the more reliable estimate.

### What drives the prediction

In [9]:
imp = permutation_importance(best, X_test, y_test, scoring='roc_auc', n_repeats=20, random_state=SEED)
pd.Series(imp.importances_mean, index=X_test.columns).sort_values().plot.barh(
    figsize=(7, 4), color='#4f46e5', title='Permutation importance (drop in ROC-AUC)');

### Try a few passengers

In [10]:
people = pd.DataFrame([
    {'Pclass': '1st', 'Sex': 'female', 'Age': 30, 'Fare': 100, 'SibSp': 1, 'Parch': 0, 'Embarked': 'Cherbourg', 'Title': 'Mrs'},
    {'Pclass': '3rd', 'Sex': 'male', 'Age': 25, 'Fare': 8, 'SibSp': 0, 'Parch': 0, 'Embarked': 'Southampton', 'Title': 'Mr'},
    {'Pclass': '3rd', 'Sex': 'male', 'Age': 6, 'Fare': 30, 'SibSp': 3, 'Parch': 2, 'Embarked': 'Southampton', 'Title': 'Master'},
    {'Pclass': '2nd', 'Sex': 'male', 'Age': 5, 'Fare': 25, 'SibSp': 1, 'Parch': 1, 'Embarked': 'Southampton', 'Title': 'Master'},
]).astype({c: float for c in ['Age', 'Fare', 'SibSp', 'Parch']})
people.assign(p_survive=best.predict_proba(people)[:, 1].round(2))

,Pclass,Sex,Age,Fare,SibSp,Parch,Embarked,Title,p_survive
0,1st,female,30.0,100.0,1.0,0.0,Cherbourg,Mrs,0.97
1,3rd,male,25.0,8.0,0.0,0.0,Southampton,Mr,0.09
2,3rd,male,6.0,30.0,3.0,2.0,Southampton,Master,0.07
3,2nd,male,5.0,25.0,1.0,1.0,Southampton,Master,0.87


## 5. Conclusions
- Sex, title and class dominate: “women and children first” and 1st-class access to the lifeboats are clearly visible.
- The title extracted from the name is the most useful engineered feature; family size and fare per person add only a little on top (+0.003 CV ROC-AUC for Logistic Regression).
- Gradient Boosting is the final model (CV ROC-AUC ≈ 0.89, accuracy ≈ 83%).
- Engineered features live inside the pipeline, so the exported ONNX model needs only what a person can type.

`train.py` contains the production version; the pipeline is exported to ONNX and runs in the web demo.